# Question Answering System â€” Google Colab Notebook

![CNN+BiLSTM+Attention](https://img.shields.io/badge/architecture-CNN%2BBiLSTM%2BAttention-blue)
![Framework](https://img.shields.io/badge/framework-PyTorch-orange)

This notebook runs the **neural extractive Question Answering** system entirely in the cloud (free GPU/CPU via Google Colab).

**Pipeline:**

```
Text â†’ Tokenization â†’ Word Embeddings â†’ CNN (n-gram) â†’ BiLSTM â†’ Attention â†’ Start/End Prediction â†’ Extract Answer
```

**What you will do here:**
1. Set up the environment and pull the project source from GitHub
2. Inspect the sample SQuAD-style dataset
3. Preprocess and build the vocabulary
4. Train the CNN + BiLSTM + Attention model (use a GPU runtime for speed)
5. Visualise training curves (loss, EM, F1)
6. Run live inference on your own context + question
7. Download the trained model to use with the local Flask/React app

> **Tip:** In the top-right menu choose **Runtime â†’ Change runtime type â†’ GPU** to speed up training dramatically.



## 0. Choose a configuration

Run this cell first. You can switch between a quick sample test and full SQuAD training.



In [ ]:
# ==== CONFIGURATION ======================================================
# Set to True to clone the repo from GitHub; False to keep /content/qasdl if it exists.
FETCH_FROM_GITHUB = True

# Change this to your GitHub repo if you forked/cloned elsewhere.
GITHUB_REPO = "jeanvil16/question-answering-dl"

# Training options (sample dataset by default)
USE_SAMPLE_DATASET = True   # True -> sample_squad.json (fast, no download)
                           # False -> full SQuAD v1.1 (downloads ~35 MB)
TRAIN_EPOCHS = 40
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
MAX_CONTEXT_LEN = 160
MAX_QUESTION_LEN = 24

# ==========================================================================



## 1. Environment setup

Colab already includes PyTorch. This cell just installs the small extra dependencies and clones the repository.



In [ ]:
import sys
import importlib.util
from pathlib import Path

def has(pkg):
    return importlib.util.find_spec(pkg) is not None

# Extra deps used by the project (torch comes preinstalled in Colab)
need = [p for p in ("numpy", "tqdm", "matplotlib", "flask", "flask_cors") if not has(p)]
if need:
    print("Installing:", need)
    !pip install -q {' '.join(need)}
else:
    print("All Python dependencies already present.")

print("PyTorch version:", __import__('torch').__version__)
import torch
print("CUDA available:", torch.cuda.is_available())



In [ ]:
import os
import shutil

PROJECT_DIR = Path("/content/qasdl")

if FETCH_FROM_GITHUB or not PROJECT_DIR.exists():
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    print(f"Cloning https://github.com/{GITHUB_REPO} ...")
    !git clone -q --depth 1 https://github.com/{GITHUB_REPO}.git {PROJECT_DIR}
else:
    print("Using existing project directory.")

os.chdir(PROJECT_DIR)
print("Working directory:", PROJECT_DIR)
print("Contents:", sorted(p.name for p in PROJECT_DIR.iterdir() if not p.name.startswith('.')))



## 2. Explore the dataset

The project ships a hand-crafted SQuAD-format dataset. If `USE_SAMPLE_DATASET` is `False`, we download the full SQuAD v1.1 instead.



In [ ]:
import json
import subprocess

if USE_SAMPLE_DATASET:
    DATA_FILE = PROJECT_DIR / "dataset" / "sample_squad.json"
    print("Using sample dataset:", DATA_FILE)
else:
    DATA_FILE = PROJECT_DIR / "dataset" / "train-v1.1.json"
    if not DATA_FILE.exists():
        print("Downloading full SQuAD v1.1 ...")
        subprocess.run(["python", "dataset/download_squad.py"], check=True)
    print("Using full SQuAD dataset:", DATA_FILE)

squad = json.load(open(DATA_FILE, encoding="utf-8"))
n_passages = len(squad["data"])
n_qas = sum(len(p["qas"]) for a in squad["data"] for p in a["paragraphs"])
print(f"Passages: {n_passages}")
print(f"Questions: {n_qas}")
print("First passage title:", squad["data"][0]["title"])
print("First question:", squad["data"][0]["paragraphs"][0]["qas"][0]["question"])
print("First answer:", squad["data"][0]["paragraphs"][0]["qas"][0]["answers"][0]["text"])



## 3. Train the model

This calls the project's training script. It:
- tokenises context + question with character-offset alignment
- builds the vocabulary
- trains the CNN + BiLSTM + Attention model with joint start/end cross-entropy loss
- validates each epoch with SQuAD EM / F1 metrics
- saves the best checkpoint + vocab + config + history to `saved_models/`



In [ ]:
print("Training the QA model ...")

if USE_SAMPLE_DATASET:
    cmd = [
        "python", "training/train.py",
        "--epochs", str(TRAIN_EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--learning-rate", str(LEARNING_RATE),
        "--max-context-len", str(MAX_CONTEXT_LEN),
        "--max-question-len", str(MAX_QUESTION_LEN),
    ]
else:
    cmd = [
        "python", "training/train.py",
        "--train-file", "dataset/train-v1.1.json",
        "--val-file", "dataset/dev-v1.1.json",
        "--epochs", "3", "--batch-size", "64",
        "--min-token-freq", "2",
        "--max-context-len", "320", "--max-question-len", "32",
    ]

print(" ".join(cmd))
import subprocess, sys as _sys
subprocess.run(cmd)



## 4. Visualise training results

Plots the loss curves and the Exact Match / F1 accuracy metrics saved during training.



In [ ]:
import json
import matplotlib.pyplot as plt

hist = json.load(open(PROJECT_DIR / "saved_models" / "training_history.json", encoding="utf-8"))
epochs = [h["epoch"] for h in hist]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2))

ax1.plot(epochs, [h["train_loss"] for h in hist], label="Train Loss", color="#60a5fa")
ax1.plot(epochs, [h["val_loss"] for h in hist], label="Val Loss", color="#f472b6")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Loss Curves"); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, [h["train_em"] for h in hist], label="Train EM", ls="--", color="#34d399")
ax2.plot(epochs, [h["val_em"] for h in hist], label="Val EM", color="#34d399")
ax2.plot(epochs, [h["train_f1"] for h in hist], label="Train F1", ls="--", color="#fb923c")
ax2.plot(epochs, [h["val_f1"] for h in hist], label="Val F1", color="#fb923c")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Score"); ax2.set_title("Accuracy: EM / F1"); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()

best = max(hist, key=lambda h: h["val_f1"])
print(f"Best epoch {best['epoch']}: val EM {best['val_em']:.2%}, val F1 {best['val_f1']:.2%}")



## 5. Live inference on your own text

Load the trained model and answer your own question. Edit the `context` and `question` variables below, then run the cell.



In [ ]:
context = (
    "The Taj Mahal is a white marble mausoleum located in Agra, India, on the "
    "southern bank of the Yamuna River. It was commissioned in 1632 by the Mughal "
    "emperor Shah Jahan in memory of his favourite wife, Mumtaz Mahal."
)
question = "Who commissioned the Taj Mahal?"

# ---- load the saved model -----------------------------------------------
from model.qa_model import ExtractiveQAModel, QAModelConfig, decode_best_span
from preprocessing.vocabulary import Vocabulary
from preprocessing.tokenizer import tokenize_with_offsets
import torch

SAVED = PROJECT_DIR / "saved_models"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = QAModelConfig.load(SAVED / "model_config.json")
vocab = Vocabulary.load(SAVED / "vocab.json")
model = ExtractiveQAModel(config).to(device)
state = torch.load(SAVED / "qa_model.pt", map_location=device, weights_only=True)
if isinstance(state, dict) and "state_dict" in state:
    state = state["state_dict"]
model.load_state_dict(state)
model.eval()

# ---- tokenize ------------------------------------------------------------
q = tokenize_with_offsets(question)[:config.max_question_len]
c = tokenize_with_offsets(context)[:config.max_context_len]
q_ids = torch.tensor([vocab.encode([t.text for t in q])], device=device)
c_ids = torch.tensor([vocab.encode([t.text for t in c])], device=device)
q_mask = torch.ones_like(q_ids, dtype=torch.bool)
c_mask = torch.ones_like(c_ids, dtype=torch.bool)

with torch.no_grad():
    start_logits, end_logits = model(q_ids, q_mask, c_ids, c_mask)

# ---- decode span + confidence --------------------------------------------
best_s, best_e = decode_best_span(start_logits[0].cpu(), end_logits[0].cpu(), config.max_answer_len)
answer = context[c[best_s].start:c[best_e].end]
probs = torch.softmax(torch.stack([start_logits[0], end_logits[0]]), dim=-1)
confidence = float((probs[0, best_s] * probs[1, best_e]) ** 0.5)

print("Question   :", question)
print("Answer     :", answer)
print(f"Confidence : {confidence:.1%}")



## 6. Download the trained model

Save the trained weights to your local machine so you can use them with the **local** Flask + React app (`backend/app.py`).

The files below should be placed in the `saved_models/` folder of the project on your computer:
```
saved_models/
â”œâ”€â”€ qa_model.pt
â”œâ”€â”€ model_config.json
â”œâ”€â”€ vocab.json
â”œâ”€â”€ training_history.json
â””â”€â”€ metrics.json
```



In [ ]:
from google.colab import files

print("Downloading model artifacts...")
for name in ["qa_model.pt", "model_config.json", "vocab.json",
             "training_history.json", "metrics.json"]:
    path = SAVED / name
    if path.exists():
        files.download(str(path))
    else:
        print(f"Missing: {name}")
print("Done. Place these in your local saved_models/ folder.")




---
## Appendix: full SQuAD training

If you set `USE_SAMPLE_DATASET = False` in the configuration cell and rerun from section 2, the notebook downloads the official [SQuAD v1.1](https://rajpurkar.github.io/SQuAD-explorer/) dataset and trains on it. Expect much longer training time; a GPU runtime is strongly recommended.



### Reference

Project repository: [github.com/jeanvil16/question-answering-dl](https://github.com/jeanvil16/question-answering-dl)

Deep learning architecture: **Embedding â†’ Multi-kernel CNN (k=2,3,4) â†’ BiLSTM â†’ BiDAF-style Attention â†’ Start/End heads**.

